# Notebook 02 — Baseline Model Evaluation

**Assignment 1B — Enterprise Domain LLM**  
**Author:** Abhinav Mandloi

In this notebook we:
1. Load **Qwen/Qwen2.5-1.5B-Instruct** (1.5 billion parameters) on MPS/CPU with bfloat16
2. Inspect the architecture (layers, hidden size, vocab, attention heads)
3. Run three IT helpdesk prompts and record timing + token counts
4. Save all outputs to `results/baseline/baseline_outputs.json`

---

In [ ]:
import os, sys, json, time
from pathlib import Path
import torch

PROJECT_ROOT = Path('/Users/abhinavmandloi/Documents/Sem3Projects/LLM Assingment 1/enterprise-domain-llm')
RESULTS_DIR  = PROJECT_ROOT / 'results' / 'baseline'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Device selection
if torch.cuda.is_available():
    DEVICE = 'cuda'
    DTYPE  = torch.float16
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
    DTYPE  = torch.bfloat16
else:
    DEVICE = 'cpu'
    DTYPE  = torch.float32

print(f'Device : {DEVICE}')
print(f'DType  : {DTYPE}')
print(f'PyTorch: {torch.__version__}')

## 2.1 Load Model & Tokenizer

We use `Qwen/Qwen2.5-1.5B-Instruct` — a 1.5B parameter instruction-tuned transformer with:
- Grouped Query Attention (GQA)
- 128K context window
- BFloat16 support (MPS-compatible)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

print(f'Loading tokenizer: {MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f'Loading model    : {MODEL_ID} ...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map=DEVICE,
    trust_remote_code=True,
)
model.eval()

print('Model loaded successfully.')
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,} ({total_params/1e9:.2f}B)')

## 2.2 Architecture Inspection

In [ ]:
import pandas as pd

cfg = model.config

arch_info = [
    ('Model ID',               MODEL_ID),
    ('Architecture',           type(model).__name__),
    ('Num hidden layers',      getattr(cfg, 'num_hidden_layers', 'N/A')),
    ('Hidden size',            getattr(cfg, 'hidden_size', 'N/A')),
    ('Intermediate size',      getattr(cfg, 'intermediate_size', 'N/A')),
    ('Num attention heads',    getattr(cfg, 'num_attention_heads', 'N/A')),
    ('Num KV heads (GQA)',     getattr(cfg, 'num_key_value_heads', 'N/A')),
    ('Vocab size',             getattr(cfg, 'vocab_size', 'N/A')),
    ('Max position embeddings',getattr(cfg, 'max_position_embeddings', 'N/A')),
    ('RoPE theta',             getattr(cfg, 'rope_theta', 'N/A')),
    ('Sliding window',         getattr(cfg, 'sliding_window', 'N/A')),
    ('Torch dtype',            str(DTYPE)),
    ('Total parameters',       f'{total_params:,}'),
]

df_arch = pd.DataFrame(arch_info, columns=['Property', 'Value'])
print('Architecture Summary:')
print(df_arch.to_string(index=False))

# Print first decoder block structure
print('\nFirst decoder layer structure:')
print(model.model.layers[0])

## 2.3 Inference: Three IT Helpdesk Prompts

We evaluate the **untuned** baseline on three representative IT troubleshooting queries:

| ID | Prompt Domain |
|----|---------------|
| P1 | Ubuntu / NTFS drive mounting |
| P2 | Linux production server I/O & CPU |
| P3 | AWS EC2 connectivity & SSH |

In [ ]:
PROMPTS = [
    {
        'id'    : 'P1',
        'prompt': 'How do I troubleshoot an Ubuntu server that cannot mount an NTFS drive?'
    },
    {
        'id'    : 'P2',
        'prompt': 'How do I investigate high disk I/O and CPU usage on a Linux production server?'
    },
    {
        'id'    : 'P3',
        'prompt': 'How do I troubleshoot AWS EC2 instance connectivity and SSH access issues?'
    },
]

MAX_NEW_TOKENS = 300

def run_inference(prompt_text: str, model, tokenizer, device: str, max_new_tokens: int = 300) -> dict:
    """Run inference with a chat template and return output + timing stats."""
    messages = [
        {'role': 'system', 'content': 'You are a helpful IT support assistant with expertise in Linux, Ubuntu, and AWS cloud infrastructure.'},
        {'role': 'user',   'content': prompt_text}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to(device)
    input_token_count = inputs['input_ids'].shape[-1]

    start = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy for baseline
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - start

    generated_ids = output_ids[0][input_token_count:]
    output_text   = tokenizer.decode(generated_ids, skip_special_tokens=True)
    output_tokens = len(generated_ids)

    return {
        'prompt'      : prompt_text,
        'output'      : output_text,
        'input_tokens': input_token_count,
        'output_tokens': output_tokens,
        'latency_s'   : round(elapsed, 3),
        'tokens_per_s': round(output_tokens / elapsed, 2),
    }

results = []
for p in PROMPTS:
    print(f"\n{'='*60}")
    print(f"Running {p['id']}: {p['prompt'][:60]}...")
    res = run_inference(p['prompt'], model, tokenizer, DEVICE, MAX_NEW_TOKENS)
    res['id'] = p['id']
    results.append(res)
    print(f"  Latency  : {res['latency_s']:.2f}s")
    print(f"  Tokens/s : {res['tokens_per_s']:.1f}")
    print(f"  Output tokens: {res['output_tokens']}")
    print(f"\n--- Output ---")
    print(res['output'])

## 2.4 Results Summary Table

In [ ]:
df_results = pd.DataFrame([{
    'ID'            : r['id'],
    'Input tokens'  : r['input_tokens'],
    'Output tokens' : r['output_tokens'],
    'Latency (s)'   : r['latency_s'],
    'Tokens/s'      : r['tokens_per_s'],
} for r in results])

print('Baseline Results:')
print(df_results.to_string(index=False))

# Save to JSON
output_path = RESULTS_DIR / 'baseline_outputs.json'
with open(output_path, 'w') as f:
    json.dump({'model': MODEL_ID, 'device': DEVICE, 'dtype': str(DTYPE),
               'max_new_tokens': MAX_NEW_TOKENS, 'results': results}, f, indent=2)

print(f'\nSaved results to {output_path}')

## 2.5 Latency Bar Chart

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ids    = [r['id'] for r in results]
lats   = [r['latency_s'] for r in results]
tps    = [r['tokens_per_s'] for r in results]

axes[0].bar(ids, lats, color=['#4C72B0', '#55A868', '#C44E52'])
axes[0].set_title('Baseline: Latency per Prompt', fontweight='bold')
axes[0].set_ylabel('Latency (s)')
axes[0].set_xlabel('Prompt ID')

axes[1].bar(ids, tps, color=['#4C72B0', '#55A868', '#C44E52'])
axes[1].set_title('Baseline: Throughput per Prompt', fontweight='bold')
axes[1].set_ylabel('Tokens / second')
axes[1].set_xlabel('Prompt ID')

plt.tight_layout()
charts_dir = PROJECT_ROOT / 'results' / 'charts'
charts_dir.mkdir(exist_ok=True)
plt.savefig(charts_dir / 'baseline_latency.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 2.6 Observations

The baseline Qwen2.5-1.5B-Instruct model demonstrates reasonable general IT knowledge but lacks:
- **Domain-specific terminology** (e.g., exact command flags for `ntfs-3g`, `iotop` options)
- **Structured troubleshooting steps** in the enterprise IT format
- **AWS-specific nuances** such as VPC routing and Security Group rules

Fine-tuning with domain-specific instruction pairs (Notebook 04) is expected to improve output quality on these exact prompts.

---
**Next:** Notebook 03 — Instruction Dataset